# Лабораторная работа № 1
## Метод вращений Якоби

### 1. Математическая постановка

Для симметричной матрицы $A=A^T$ ищем $V$ и диагональную $\Lambda$:

$$AV=V\Lambda, \qquad V^TV=I, \qquad A=V\Lambda V^T.$$

Столбец $j$ матрицы $V$ — собственный вектор для $\lambda_j$.
Повороты Якоби последовательно уменьшают внедиагональную часть:

$$A_{k+1}=P_k^TA_kP_k,\qquad V_{k+1}=V_kP_k,\quad V_0=I.$$

Используем циклический обход пар $p<q$ и критерий
$S=\sum_{i\ne j}a_{ij}^2<\varepsilon$.

> В реализации норма пересчитывается после каждого прохода.
>
> По умолчанию (`epsilon=None`) используется относительный порог
> $\|\mathrm{offdiag}(A)\|_F<\varepsilon_{\text{маш}}\|A\|_F$: абсолютный порог $S<\varepsilon$
> зависит от масштаба матрицы.


In [1]:
from pathlib import Path
import sys
import numpy as np

for root in (Path("Lab1"), Path(".")):
    if (root / "jacobi_lab").is_dir():
        sys.path.insert(0, str(root.resolve()))
        break
from jacobi_lab import jacobi_eigh, jacobi_rotation, JacobiError
np.set_printoptions(precision=10, suppress=True, linewidth=100)


### 2. Матрица из примера

Используем пример 5×5. Элементы задаются формулой
$a_{ij}=1+3(i+j)$ при индексации с нуля. Матрица симметрична и имеет ранг 2.


In [2]:
A = np.array([[1 + 3*(i+j) for j in range(5)] for i in range(5)], dtype=float)
print(A)
result = jacobi_eigh(A, max_sweeps=100)
print("\nСобственные числа:")
print(result.eigenvalues)
print("\nСобственные векторы (столбцы):")
print(result.eigenvectors)
print(f"\nПроходов: {result.sweeps}; вращений: {result.rotations}")


[[ 1.  4.  7. 10. 13.]
 [ 4.  7. 10. 13. 16.]
 [ 7. 10. 13. 16. 19.]
 [10. 13. 16. 19. 22.]
 [13. 16. 19. 22. 25.]]

Собственные числа:
[-6.3104367407 -0.           -0.            0.           71.3104367407]

Собственные векторы (столбцы):
[[ 0.7337151707  0.0797762332 -0.497465374  -0.3823139472  0.2483184412]
 [ 0.4306142293 -0.6099606009  0.5170337258  0.2462198037  0.3384839517]
 [ 0.1275132878  0.7556288341  0.4743173026  0.0636251644  0.4286494622]
 [-0.1755876537 -0.0004807984 -0.5098742866  0.663346049   0.5188149727]
 [-0.4786885951 -0.224963668   0.0159886322 -0.5908770699  0.6089804832]]

Проходов: 4; вращений: 40


### 3. Один шаг метода

Воспроизведём численный пример: зануляем элемент $A_{24}$
(в нумерации с нуля — позиция $(1, 3)$). По формулам:
$C=6/13$, $\operatorname{tg}\varphi\approx0.64$, $\cos\varphi\approx0.842$, $\sin\varphi\approx0.539$, и получаем матрицу $A_{i+1}$.

In [3]:
step = jacobi_rotation(A, 1, 3)
print(f"tg φ = {step.tan:.4f}, cos φ = {step.cos:.4f}, sin φ = {step.sin:.4f}")
print("\nA_{i+1} (округлено до 0.01):")
print(np.round(step.matrix, 2))
book = np.array([[1, -2.02, 7, 10.58, 13], [-2.02, -1.32, -0.2, 0, 1.62],
                 [7, -0.2, 13, 18.86, 19], [10.58, 0, 18.86, 27.32, 27.15],
                 [13, 1.62, 19, 27.15, 25]])
print(f"\nМакс. отличие от матрицы в примере: {np.max(np.abs(step.matrix - book)):.3f}")
P = np.eye(5)
P[1, 1] = P[3, 3] = step.cos
P[1, 3], P[3, 1] = step.sin, -step.sin
print(f"Отличие от явного произведения PᵀAP: {np.max(np.abs(step.matrix - P.T @ A @ P)):.1e}")


tg φ = 0.6398, cos φ = 0.8423, sin φ = 0.5390

A_{i+1} (округлено до 0.01):
[[ 1.   -2.02  7.   10.58 13.  ]
 [-2.02 -1.32 -0.2   0.    1.62]
 [ 7.   -0.2  13.   18.87 19.  ]
 [10.58  0.   18.87 27.32 27.15]
 [13.    1.62 19.   27.15 25.  ]]

Макс. отличие от матрицы в примере: 0.007
Отличие от явного произведения PᵀAP: 3.6e-15


### 4. Проверка правильности

Сравниваем спектр с независимой библиотечной реализацией. Проверяем невязку,
ортонормированность и восстановление исходной матрицы. Собственные векторы
напрямую не сравниваем: их знаки произвольны, а для кратного нулевого собственного
числа возможны разные ортонормированные базисы.

Для этой матрицы точные собственные числа:

$$\lambda_{1,5}=\frac{65\mp\sqrt{6025}}{2},\qquad
\lambda_2=\lambda_3=\lambda_4=0.$$


In [4]:
w, V = result.eigenvalues, result.eigenvectors
reference = np.linalg.eigvalsh(A)
exact = np.array([(65-np.sqrt(6025))/2, 0, 0, 0, (65+np.sqrt(6025))/2])
metrics = {
    "Невязка ||AV − VΛ||F": np.linalg.norm(A @ V - V*w),
    "Ортогональность ||VᵀV − I||F": np.linalg.norm(V.T @ V - np.eye(len(A))),
    "Восстановление ||A − VΛVᵀ||F": np.linalg.norm(A - (V*w) @ V.T),
    "Макс. отклонение от NumPy": np.max(np.abs(w-reference)),
    "Макс. отклонение от точного спектра": np.max(np.abs(w-exact)),
}
for title, value in metrics.items():
    print(f"{title:42s} {value:.6e}")
assert np.allclose(A @ V, V*w, atol=1e-9)
assert np.allclose(V.T @ V, np.eye(len(A)), atol=1e-12)
print("\nВсе проверки примера пройдены.")


Невязка ||AV − VΛ||F                       2.753583e-14
Ортогональность ||VᵀV − I||F               1.864844e-15
Восстановление ||A − VΛVᵀ||F               6.112733e-14
Макс. отклонение от NumPy                  1.421085e-14
Макс. отклонение от точного спектра        1.421085e-14

Все проверки примера пройдены.


### 5. Сходимость

Здесь показана норма внедиагональной части в исходном масштабе и сумма
$S$ её квадратов. Итерации останавливаются, когда норма становится меньше
порога (по умолчанию $\varepsilon_{\text{маш}}\|A\|_F$).

In [5]:
print(f"Порог для нормы: {result.tolerance_scaled * result.scale:.3e}")
print(f"{'Проход':>8} {'||offdiag(Ak)||F':>22} {'S':>22}")
for sweep, scaled_norm in enumerate(result.history):
    norm = scaled_norm * result.scale
    print(f"{sweep:8d} {norm:22.8e} {norm**2:22.8e}")


Порог для нормы: 1.590e-14
  Проход       ||offdiag(Ak)||F                      S
       0         6.26099034e+01         3.92000000e+03
       1         6.85905778e+00         4.70466737e+01
       2         9.30485970e-01         8.65804140e-01
       3         1.22481426e-07         1.50016997e-14
       4         2.81399511e-16         7.91856848e-32


### 6. Особые случаи и контролируемые ошибки

При равных диагональных элементах нужен отдельный случай $C=0$.
Нулевая матрица возвращается без вращений. Лимит проходов исключает зацикливание.

Последний пример показывает, почему порог по умолчанию относительный: у матрицы
$10^{-12}\begin{pmatrix}2&1\\1&2\end{pmatrix}$ сумма $S=2\cdot10^{-24}$ уже меньше
$\varepsilon=10^{-20}$, поэтому абсолютный критерий останавливает метод до первого
поворота и возвращает диагональ $(2, 2)\cdot10^{-12}$ вместо спектра $(1, 3)\cdot10^{-12}$.

In [6]:
for name, matrix in [("Равная диагональ", [[2, 1], [1, 2]]),
                     ("Нулевая матрица", np.zeros((3, 3)))]:
    r = jacobi_eigh(matrix)
    print(f"{name}: λ={r.eigenvalues}; вращений={r.rotations}")
for name, matrix, options in [
    ("Нарушение симметрии", [[1, 2], [3, 4]], {}),
    ("Лимит проходов", [[2, 1], [1, 2]], {"max_sweeps": 0}),
]:
    try:
        jacobi_eigh(matrix, **options)
    except JacobiError as error:
        print(f"{name}: {type(error).__name__}: {error}")

small = np.array([[2., 1.], [1., 2.]]) * 1e-12
for label, options in [("epsilon=1e-20 (абсолютный)", {"epsilon": 1e-20}),
                       ("по умолчанию (относительный)", {})]:
    r = jacobi_eigh(small, **options)
    print(f"Малая матрица, {label}: λ={r.eigenvalues.tolist()}; вращений={r.rotations}")
print("Эталон NumPy:", np.linalg.eigvalsh(small).tolist())


Равная диагональ: λ=[1. 3.]; вращений=1
Нулевая матрица: λ=[0. 0. 0.]; вращений=0
Нарушение симметрии: InvalidInputError: Матрица должна быть симметричной: A = A.T.
Лимит проходов: ConvergenceError: Точность не достигнута за 0 проходов; норма/масштаб=7.071e-01, порог/масштаб=3.511e-16.
Малая матрица, epsilon=1e-20 (абсолютный): λ=[2e-12, 2e-12]; вращений=0
Малая матрица, по умолчанию (относительный): λ=[1e-12, 3e-12]; вращений=1
Эталон NumPy: [9.999999999999998e-13, 3e-12]


### 7. Выводы

Метод Якоби реализован через обновление двух строк/столбцов без полного
матричного умножения на каждом повороте. Отдельный шаг совпадает с численным
примером. Для примера получены два ненулевых
собственных числа (приблизительно −6.3104367407 и 71.3104367407) и три числа,
близких к нулю в пределах погрешности округления. Численные проверки подтверждают
спектральное разложение и ортонормированность найденного базиса.

Программа оформлена как Python-пакет с публичным интерфейсом, проверками
предусловий и постусловий, консольной командой и набором тестов. Теоретические
свойства подтверждены на аналитических и случайных матрицах. Порог остановки по умолчанию
относительный, поэтому результат корректен для матриц любого масштаба; абсолютный
критерий $S<\varepsilon$ доступен через параметр `epsilon`. Точность ограничена
арифметикой float64; для задач за её пределами предусмотрена диагностическая ошибка.

**Запуск всех тестов из папки Lab1:** `python -m unittest discover -s tests -v`.
Полное описание API, ограничений и установки находится в `README.md`.
